In [1]:
#this code with reload any imports if you're updating them actively
%reload_ext autoreload
%autoreload 2
import pygem.pygem_input as pygem_prms
import scipy.optimize as opt
import numpy as np
import xarray as xr
import pandas as pd
import os
import suncalc as solar
#PyGEM
import pygem.pygem_input as pygem_prms
import pygem.oggm_compat as oggm
import pygem.pygem_modelsetup as modelsetup
import class_climate

In [2]:
# # read 10 year data files for each variable in varnames and merge
# eb_varnames = ['temp','dtemp','precip','surfrad','tcc','uwind','vwind']
# i=2
# varname = 'press'
# hourlyfp = '~/research/climate_data/ERA5/ERA5_hourly/varname/ERA5_varname_hourly'.replace('varname',varname)
# da_0 = xr.open_dataarray(hourlyfp + '_80_89.nc')
# da_1 = xr.open_dataarray(hourlyfp + '_90_99.nc')
# da_2 = xr.open_dataarray(hourlyfp + '_00_09.nc')
# da_3 = xr.open_dataarray(hourlyfp + '_10_21.nc')
# print(da_0.coords)
# print(da_1.coords)
# print(da_2.coords)
# print(da_3.coords)

# da = xr.merge([da_0,da_1,da_2,da_3])
# da.to_netcdf('~/research/climate_data/ERA5/ERA5_hourly/ERA5_varname_hourly.nc'.replace('varname',varname))
# print(da)
# #368,164

In [3]:
gcm = class_climate.GCM(name='ERA5-hourly')
if pygem_prms.glac_no not in ['01.00570'] and pygem_prms.run_eb:
    print('EB model can currently only run Gulkana glacier')
main_glac_rgi = modelsetup.selectglaciersrgitable(pygem_prms.glac_no,
                rgi_regionsO1=pygem_prms.rgi_regionsO1, rgi_regionsO2=pygem_prms.rgi_regionsO2,
                rgi_glac_number=pygem_prms.rgi_glac_number, include_landterm=pygem_prms.include_landterm,
                include_laketerm=pygem_prms.include_laketerm, include_tidewater=pygem_prms.include_tidewater)
# ===== TIME PERIOD =====
startyr = 1980
endyr = 2021
dates_table = modelsetup.datesmodelrun(startyear=startyr, endyear=endyr, spinupyears=pygem_prms.gcm_spinupyears,
            option_wateryear=pygem_prms.gcm_wateryear)

EB model can currently only run Gulkana glacier
1 glaciers in region 1 are included in this model run: ['00570']
This study is focusing on 1 glaciers in region [1]
!! Not handling water years in modelsetup.datesmodelrun


In [4]:
%load_ext line_profiler

In [28]:
%reload_ext autoreload
%autoreload 2
import class_climate
gcm = class_climate.GCM(name='ERA5-hourly')
%lprun -f gcm.importGCMvarnearestneighbor_xarray gcm.importGCMvarnearestneighbor_xarray(gcm.vwind_fn, gcm.vwind_vn, main_glac_rgi,dates_table)

Timer unit: 1e-09 s

Total time: 0.0201188 s
File: /home/claire/research/PyGEM/class_climate.py
Function: importGCMvarnearestneighbor_xarray at line 264

Line #      Hits         Time  Per Hit   % Time  Line Contents
   264                                               def importGCMvarnearestneighbor_xarray(self, filename, vn, main_glac_rgi, dates_table, realizations=['r1i1p1f1','r4i1p1f1']):
   265                                                   """
   266                                                   Import time series of variables and extract nearest neighbor.
   267                                                   
   268                                                   Note: "NG" refers to a homogenized "new generation" of products from ETH-Zurich.
   269                                                         The function is setup to select netcdf data using the dimensions: time, latitude, longitude (in that 
   270                                                         

In [10]:
gcm_temp_1 = xr.open_dataset('~/research/climate_data/ERA5/ERA5_hourly/ERA5_temp_hourly.nc')
print(gcm_temp_1.t2m)
gcm_temp, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.temp_fn, gcm.temp_vn, main_glac_rgi,dates_table)
print(gcm_temp)

<xarray.DataArray 't2m' (time: 368184, latitude: 1, longitude: 1)>
[368184 values with dtype=float32]
Coordinates:
  * longitude  (longitude) float32 -145.5
  * latitude   (latitude) float32 63.34
  * time       (time) datetime64[ns] 1980-01-01 ... 2021-12-31T23:00:00
Attributes:
    units:      K
    long_name:  2 metre temperature
[[-33.46214905 -33.56300964 -33.94701233 ... -16.48670044 -16.2444519
  -15.86252441]]


In [26]:
# ===== LOAD CLIMATE DATA =====
gcm_prec, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.prec_fn, gcm.prec_vn, main_glac_rgi,dates_table)
gcm_temp, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.temp_fn, gcm.temp_vn, main_glac_rgi,dates_table)
gcm_dtemp, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.dtemp_fn, gcm.dtemp_vn, main_glac_rgi,dates_table)
#gcm_press, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.press_fn, gcm.press_vn, main_glac_rgi,dates_table)
gcm_tcc, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.tcc_fn, gcm.tcc_vn, main_glac_rgi,dates_table)
gcm_surfrad, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.surfrad_fn, gcm.surfrad_vn, main_glac_rgi,dates_table) 
gcm_uwind, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.uwind_fn, gcm.uwind_vn, main_glac_rgi,dates_table)                                                      
gcm_vwind, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.vwind_fn, gcm.vwind_vn, main_glac_rgi,dates_table)

gcm_elev = gcm.importGCMfxnearestneighbor_xarray(gcm.elev_fn, gcm.elev_vn, main_glac_rgi)

gcm_elev being handled stupidly for EB, revisit later


In [6]:
# READ GULKANA ELEVATION DATA FROM RGI 6.0
gulkana = '01.00570'
glac_no = [gulkana]

# ----- RGI DATA -----
# Filepath for RGI files
main_directory = os.getcwd()
print('Gulkana Glacier Table:')
print(main_glac_rgi)
elev_points = main_glac_rgi[['Zmin','Zmed','Zmax']]

1 glaciers in region 1 are included in this model run: ['00570']
This study is focusing on 1 glaciers in region [1]
   O1Index           RGIId   CenLon  CenLat  O1Region  O2Region    Area  Zmin  \
0      569  RGI60-01.00570 -145.427  63.281         1         2  17.567  1162   

   Zmax  Zmed  ...  Lmax  Connect  Form  TermType  Surging   RefDate   glacno  \
0  2438  1858  ...  8639        0     0         0        9  20090804  1.00570   

   rgino_str RGIId_float CenLon_360  
0   01.00570      1.0057    214.573  

[1 rows x 22 columns]
   O1Index           RGIId   CenLon  CenLat  O1Region  O2Region    Area  Zmin  \
0      569  RGI60-01.00570 -145.427  63.281         1         2  17.567  1162   

   Zmax  Zmed  ...  Lmax  Connect  Form  TermType  Surging   RefDate   glacno  \
0  2438  1858  ...  8639        0     0         0        9  20090804  1.00570   

   rgino_str RGIId_float CenLon_360  
0   01.00570      1.0057    214.573  

[1 rows x 22 columns]
Gulkana Glacier Table:
   O1Index 

In [5]:
#%% MODEL PROPERTIES
density_ice = 900           # Density of ice [kg m-3] (or Gt / 1000 km3)
density_water = 1000        # Density of water [kg m-3]
area_ocean = 362.5 * 1e12   # Area of ocean [m2] (Cogley, 2012 from Marzeion et al. 2020)
k_ice = 2.33                # Thermal conductivity of ice [J s-1 K-1 m-1] recall (W = J s-1)
k_air = 0.023               # Thermal conductivity of air [J s-1 K-1 m-1] (Mellor, 1997)
k_air = 0.001               # Thermal conductivity of air [J s-1 K-1 m-1]
ch_ice = 1890000            # Volumetric heat capacity of ice [J K-1 m-3] (density=900, heat_capacity=2100 J K-1 kg-1)
ch_air = 1297               # Volumetric Heat capacity of air [J K-1 m-3] (density=1.29, heat_capacity=1005 J K-1 kg-1)
Lh_rf = 333550              # Latent heat of fusion [J kg-1]
tolerance = 1e-12           # Model tolerance (used to remove low values caused by rounding errors)
gravity = 9.81              # Gravity [m s-2]
pressure_std = 101325       # Standard pressure [Pa]
temp_std = 288.15           # Standard temperature [K]
R_gas = 8.3144598           # Universal gas constant [J mol-1 K-1]
molarmass_air = 0.0289644   # Molar mass of Earth's air [kg mol-1]


# MORE FACTORS THAT MIGHT BE NECESSARY
kp = 1                              # precipitation factor [-] (referred to as k_p in Radic etal 2013; c_prec in HH2015)
tbias = 5                           # temperature bias [deg C]
ddfsnow = 0.0041                    # degree-day factor of snow [m w.e. d-1 degC-1]
ddfsnow_iceratio = 0.7              # Ratio degree-day factor snow snow to ice
ddfice = ddfsnow / ddfsnow_iceratio # degree-day factor of ice [m w.e. d-1 degC-1]
precgrad = 0.0001                   # precipitation gradient on glacier [m-1]
lapserate = -0.0065                 # temperature lapse rate for both gcm to glacier and on glacier between elevation bins [K m-1]
tsnow_threshold = 1                 # temperature threshold for snow [deg C] (HH2015 used 1.5 degC +/- 1 degC)
calving_k = 0.7                     # frontal ablation rate [yr-1]

In [6]:
#READ GULKANA ELEVATIONS FROM OGGM GDIRS
#this step is specific to the EB for three points on Gulkana
#to generalize, need a variable geometry containing zwh for the points for the EB
gdir = oggm.single_flowline_glacier_directory(glac_no[0], logging_level='CRITICAL')
fls = oggm.get_glacier_zwh(gdir)
#filter out zero bins to get only initial glacier volume
fls = fls.iloc[np.nonzero(fls['h'].to_numpy())]
z_stats = np.array([np.min(fls['z']),np.median(fls['z']),np.max(fls['z'])])

print('Min, median and max bin elevations from OGGM flowlines:')
print(pd.Series(z_stats,index=['Zmin','Zmed','Zmax']))
print('Min, median and max elevation from RGI:')
print(elev_points)

#setup three points at minimum, median and maximum elevation band from OGGM
median_index = np.where(fls['z']==z_stats[1])[0][0]
w_stats = np.array([fls['w'][len(fls)-1],fls['w'][median_index],fls['w'][0]])
h_stats = np.array([fls['h'][len(fls)-1],fls['h'][median_index],fls['h'][0]])
geo_index = ['Bottom','Middle','Top']
geometry = pd.DataFrame({'z':z_stats,'w':w_stats,'h':h_stats},index=geo_index)

Min, median and max bin elevations from OGGM flowlines:
Zmin    1172.809436
Zmed    1628.401286
Zmax    2330.675374
dtype: float64
Min, median and max elevation from RGI:
   Zmin  Zmed  Zmax
0  1162  1858  2438


In [7]:
#manually set number of exponentially scaling bins
n_vert_bins = 10
n_points = len(geo_index)
option_bin = 0

#create variable to store glacier geometry
vert_bins = xr.Dataset(data_vars = dict(
    bin_w = (['pt'],geometry['w']),
    bin_z = (['pt'],geometry['z'])),
    coords=dict(
        point=(['pt'],geo_index),
        vert_idx=range(n_vert_bins)
        )
    )

bin_depths = np.zeros((2,n_points,n_vert_bins))
#fill vertical bin heights based on ice thickness
for g in range(n_points):
    #get ice thickness of current point
    pt_h = geometry['h'].iloc[g]
    
    if option_bin==0:
        hs = [0.1,.25,.5,.75,1,2,5,10,20,pt_h-39.6]
        ds = [sum(hs[:i]) for i in range(n_vert_bins)]
        bin_depths[:,g,:] = [hs,ds]
    else:
        c = opt.fsolve(lambda c: pt_h-np.sum(np.exp(np.arange(n_vert_bins)*c)),10)
        bin_depths[g,:] = np.exp(c*range(1,n_vert_bins))
vert_bins['bin_h'] = (['pt','vert_idx'],bin_depths[0,:,:])
vert_bins['bin_d'] = (['pt','vert_idx'],bin_depths[1,:,:])

#set bin content as a constant snow, firn or ice (s,f,i)
content_arr = np.empty((n_points,n_vert_bins),dtype=str)
content_arr[0,:] = ['i']*n_vert_bins
content_arr[1,:] = ['f']*round(n_vert_bins*.3)+['i']*round(n_vert_bins*.7)
content_arr[2,:] = ['s']*round(n_vert_bins*.2)+['f']*round(n_vert_bins*.2)+['i']*round(n_vert_bins*.6)
vert_bins['bin_content'] = (['pt','vert_idx'],content_arr)

#since using constant bin content, can also use constant properties of snow/ice
vert_bins['lambdas'] = (['pt','vert_idx'],np.where(content_arr=='s',0.9,.8))
vert_bins['rs'] = (['pt','vert_idx'],np.where(content_arr=='s',17.1,2.5))

In [8]:
#create dataset for variables that need to be adjusted
pt_climate_ds = xr.Dataset(data_vars = dict(
    bin_temp = (['pt','time'],np.zeros((n_points,len(gcm_hours)))),
    bin_prec = (['pt','time'],np.zeros((n_points,len(gcm_hours)))),
    bin_press = (['pt','time'],np.zeros((n_points,len(gcm_hours)))),
    bin_RH = (['pt','time'],np.zeros((n_points,len(gcm_hours)))),
    bin_snow = (['pt','time'],np.zeros((n_points,len(gcm_hours)))),
    bin_elev = (vert_bins['bin_z'])),
    coords=dict(
        point=(['pt'],geo_index),
        vert_idx=range(n_vert_bins),
        time=gcm_hours
        ),
    attrs=dict(description="Climate data adjusted for points in EB."))

# adjust bin temperature and precipitation using linear scaling
lr = pygem_prms.lapserate
temp_adj = [list(gcm_temp[0] + lr*(gcm_elev[0]-z)) for z in pt_climate_ds['bin_elev'].values]
prec_adj = [list(gcm_prec[0]*kp*(1+precgrad*(gcm_elev[0]-z))) for z in pt_climate_ds['bin_elev'].values]
press_adj = [list(gcm_press[0]*((gcm_temp[0] + lr*(gcm_elev[0]-z))/gcm_temp[0])**(gravity*molarmass_air/(R_gas*lr))) for z in pt_climate_ds['bin_elev'].values]
pt_climate_ds['bin_temp'] = (['pt','time'],list(temp_adj))
pt_climate_ds['bin_prec'] = (['pt','time'],list(prec_adj))
pt_climate_ds['bin_press'] = (['pt','time'],list(press_adj))
# calculate RH from temp and dewpoint temp
e_func = lambda T: 6.1078*np.exp(17.1*T/(235+T)) #vapor pressure in hPa
pt_climate_ds['bin_RH'] = (['pt','time'],list(e_func(temp_adj)/e_func(gcm_dtemp)))
pt_climate_ds['bin_snow'] = (['pt','time'],np.where(np.array(temp_adj)<(tsnow_threshold+273),1,0))

climate_ds = xr.Dataset(data_vars = dict(
    dtemp = (['glacier','time'],gcm_dtemp),
    surfrad = (['glacier','time'],gcm_surfrad),
    tcc = (['glacier','time'],gcm_tcc),
    uwind = (['glacier','time'],gcm_uwind),
    vwind = (['glacier','time'],gcm_vwind),
),coords = dict(glacier=glac_no,time=gcm_hours))

In [13]:
melt_monthly = []
time_idx = 0 #initiate time index to scale with loop
# treat as constants now, add parameterizations later:
# these are randomly thrown in values they don't mean anything
albedo = .5 
vp = 1 #vapor pressure of water: depends on the temperature
cpw = 1 #heat capacity of water
cpa = 1 #heat capacity of air

# ===== ENTER HOURLY LOOP!! =====
for hour in gcm_hours[0:10]:
    if time_idx<1:
        hourly_Q = []
        pass # need to calculate monthly_Q before can append to melt array
    elif hour.is_month_start and hour.hour > 1:
        # convert previous month Q to M, summing hourly Q_melts
        monthly_M = np.sum(hourly_Q)*3600/(density_water*Lh_rf)
        melt_monthly.append(monthly_M)
        hourly_Q = []

    for i in range(len(glac_no)):
        Ts = 0
        #some sort of while loop to check if Ts is less than 0
        result = solar.get_position(hour,glacier_table['CenLon'],glacier_table['CenLat'])
        Snet_surf = gcm_surfrad[i][time_idx] * (1-albedo) #* (cos(theta))
        #Snet = Snet_surf*vert_bins['lambdas']*np.exp(-vert_bins['bin_z']*vert_bins['rs'])
        
        #parameterization for Lnet from COSIPY
        Ecs = .23+ .433*(vp/(gcm_temp[i][time_idx]+273.15))**(1/8)
        Lnet = Ecs*(1-gcm_tcc[i][time_idx]**2)+.984*gcm_tcc[i][time_idx]**2

        w10 = np.sqrt(gcm_uwind[i][time_idx]**2+gcm_vwind[i][time_idx]**2)
        rain_mask = -(pt_climate_ds['bin_snow']-1)
        Qp = rain_mask*cpw*(gcm_temp[i][time_idx]-Ts)*gcm_prec[i][time_idx]

        k0 = 0.4
        zm = 10
        zv = zh = 2
        if snow:
            z0m = 0.001
            z0v = 0.001
            z0h = 0.001
        elif ice:
            z0m = 0.016
            z0v = 0.004
            z0v = 0.004
        kH = k0**2/(np.log(zm/z0m)*np.log(zv/z0v))
        kE = k0**2/(np.log(zm/z0m)*np.log(zh/z0h))
        density_air = gcm_press[i][time_idx]/R_gas/gcm_temp[i][time_idx]
        Qs = density_air*gcm_press[0][time_idx]*cpa*kH*w10*(gcm_temp[i][time_idx]-Ts)/pressure_std
        Ql = 0.622*density_air*kE*w10*Lv*()

        Qm = Snet_surf + Lnet + Qp
        hourly_Q.append(Qm)

    time_idx +=1


<xarray.DataArray (pt: 3, vert_idx: 10)>
array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
Coordinates:
    point     (pt) <U6 'Bottom' 'Middle' 'Top'
  * vert_idx  (vert_idx) int64 0 1 2 3 4 5 6 7 8 9
Dimensions without coordinates: pt
<xarray.DataArray (pt: 3, vert_idx: 10)>
array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
Coordinates:
    point     (pt) <U6 'Bottom' 'Middle' 'Top'
  * vert_idx  (vert_idx) int64 0 1 2 3 4 5 6 7 8 9
Dimensions without coordinates: pt
<xarray.DataArray (pt: 3, vert_idx: 10)>
array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
Coordinates:
    point     (pt) <U6 'Bottom' 'Middle' 'Top'
  * vert_idx  (vert_idx) int64 0 1 2 3 4 5 6 7 8 9
Dimensions without coordinates

In [23]:
print(melt_monthly)

[<xarray.DataArray (pt: 3, vert_idx: 10, time: 368161)>
array([[[1.33393037, 1.33393037, 1.33393037, ..., 1.33393037,
         1.33393037, 1.33393037],
        [1.33393037, 1.33393037, 1.33393037, ..., 1.33393037,
         1.33393037, 1.33393037],
        [1.33393037, 1.33393037, 1.33393037, ..., 1.33393037,
         1.33393037, 1.33393037],
        ...,
        [1.33393037, 1.33393037, 1.33393037, ..., 1.33393037,
         1.33393037, 1.33393037],
        [1.33393037, 1.33393037, 1.33393037, ..., 1.33393037,
         1.33393037, 1.33393037],
        [1.33393037, 1.33393037, 1.33393037, ..., 1.33393037,
         1.33393037, 1.33393037]],

       [[1.33393037, 1.33393037, 1.33393037, ..., 1.33393037,
         1.33393037, 1.33393037],
        [1.33393037, 1.33393037, 1.33393037, ..., 1.33393037,
         1.33393037, 1.33393037],
        [1.33393037, 1.33393037, 1.33393037, ..., 1.33393037,
         1.33393037, 1.33393037],
...
        [1.33393037, 1.33393037, 1.33393037, ..., 1.33393037,

In [16]:
monthly_ex = pd.DataFrame(monthly_M.sum(axis=1))
print(monthly_ex)
monthly_ex.to_csv('monthly_M.csv')

   0       1       2       3       4       5       6       7       8       \
0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
1     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   

   9       ...  368151  368152  368153  368154  368155  368156  368157  \
0     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
1     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2     0.0  ...     0.0     0.0     0.0     0.0     0.0     0.0     0.0   

   368158  368159  368160  
0     0.0     0.0     0.0  
1     0.0     0.0     0.0  
2     0.0     0.0     0.0  

[3 rows x 368161 columns]


In [ ]:
import pygem.pygem_input as pygem_prms
import pickle

In [ ]:
glacier_str = glac_no[0][1::]
modelprms_fn =  glacier_str + '-modelprms_dict.pkl'
modelprms_fp = pygem_prms.output_filepath + 'calibration/' + glacier_str.split('.')[0].zfill(2) + '/'
modelprms_fullfn = modelprms_fp + modelprms_fn
assert os.path.exists(modelprms_fullfn), glacier_str + ' calibrated parameters do not exist.'            
with open(modelprms_fullfn, 'rb') as f:
    modelprms_dict = pickle.load(f)